# Notebook 02 — Tracer Bullet: Persona → Match → Lay Statement

The thinnest possible end-to-end thread through the whole agent pipeline. One persona, one Diagnostic Code (DC 5260), one drafted Lay Statement that passes the Cypher-backed factuality check.

**Prereqs:**
- `make up` — Neo4j running
- `OPENAI_API_KEY` set in `.env` (used for embeddings + the drafter)
- Notebook 01 has been run at least once so DC 5260 is in the graph (or we ingest it here)

**What the tracer proves:** the pipeline can take a veteran's plain-language reports, vector-search the CFR criteria, traverse to candidate Diagnostic Codes, draft a CFR-vocabulary Lay Statement, and verify (deterministically, via Cypher) that every clinical fact in the draft traces to a confirmed report. No fabrication possible.

## 1. Make sure DC 5260 is in the graph

If notebook 01 already ran, this is a no-op. Otherwise it ingests DC 5260.

In [1]:
from va_agent.graph.driver import GraphDriver
from va_agent.ingestion.cfr import ingest_diagnostic_code

driver = GraphDriver.from_env()

rows = driver.cfr_read('MATCH (dc:CFR:DiagnosticCode {code: "5260"}) RETURN dc.code AS code')
if not rows:
    print('DC 5260 not in graph yet — running ingestion…')
    ingest_diagnostic_code(section='4.71a', dc_code='5260', driver=driver)
else:
    print('DC 5260 already in graph.')

DC 5260 not in graph yet — running ingestion…


## 2. Backfill Criterion embeddings + create the vector index

Criterion nodes carry an `embedding` property (1536-dim, OpenAI `text-embedding-3-small`). The vector index over them is what the Match Retriever queries against.

In [2]:
from va_agent.embeddings import backfill_criterion_embeddings, ensure_criterion_vector_index

ensure_criterion_vector_index(driver)
n = backfill_criterion_embeddings(driver)
print(f'Embedded {n} new Criterion nodes.')

Embedded 4 new Criterion nodes.


## 3. Record the persona's reports

The persona: Army Reservist, recently separated, MOS 15T (crew chief). He says his right knee bends only partway and locks up when he tries to crouch. He has a recent measurement of 40° flexion from a doctor's visit. On bad days, he can't kneel at all and can't run more than half a mile.

Notice the two-tier severity capture: **Baseline** and **Flare-up** — codifying the Worst-Day Rule (§4.40/§4.45).

In [3]:
from va_agent.graph.tools import (
    record_measurement,
    record_symptom,
    record_veteran,
    reset_user,
)

user_id = 'tracer-persona-army-15t'
reset_user(driver, user_id)

record_veteran(
    driver,
    user_id,
    branch='Army',
    discharge_characterization='Honorable',
    deployments=['Operation Iraqi Freedom'],
)

record_symptom(
    driver,
    user_id,
    text='my right knee only bends partway and locks up when I try to crouch',
    body_part='knee',
    typical_severity='moderate',
    flareup_severity='severe',
    flareup_frequency='weekly',
    flareup_duration='2-3 days',
    functional_loss=['cannot kneel', 'cannot run more than half a mile'],
)

record_measurement(
    driver,
    user_id,
    name='flexion',
    body_part='knee',
    value=40,
    unit='degrees',
    source='private-medical-record',
)

print('Persona recorded.')

Persona recorded.


## 4. Run the Match Retriever (Pattern 3: vector + graph traversal)

1. Embed each of the veteran's reports
2. Vector-search the Criterion nodes (top-k per report)
3. Graph-traverse from each matched Criterion to its Diagnostic Code
4. For each candidate DC, also check which Measurement thresholds the veteran's reported value satisfies
5. Rank by best-supporting Rating Percentage

In [4]:
from va_agent.retrieval.matcher import find_candidate_dcs

candidates = find_candidate_dcs(driver, user_id, top_k_per_report=5, similarity_floor=0.40)
for c in candidates:
    print(f'DC {c.code} — {c.title}  best-supported: {c.best_percent}%')
    for h in c.criterion_hits[:3]:
        print(f'    ↳ criterion: "{h.text}"  similarity={h.score:.3f}')
    for m in c.matching_measurements:
        print(f'    ↳ measurement match: vet={m["vet_value"]}{m["vet_unit"]} satisfies {m["operator"]} {m["threshold"]} → {m["supports_percent"]}%')

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_SYMPTOM` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=22, offset=125>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 125, 'line': 3, 'column': 22}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (dc:CFR:DiagnosticCode {code: $code})-[:HAS_RATING]->(rl)-[:REQUIRES]->(c:Criterion)\n                  -[:HAS_SYMPTOM]->(s:Symptom)\n            WHERE s.body_part IN $parts\n            RETURN s.name AS name, s.body_part AS body_part, rl.percent AS percent\n            '


DC 5260 — Leg, limitation of flexion of  best-supported: 30%
    ↳ criterion: "Flexion limited to 45°"  similarity=0.847
    ↳ criterion: "Flexion limited to 60°"  similarity=0.845
    ↳ criterion: "Flexion limited to 30°"  similarity=0.839
    ↳ measurement match: vet=40.0degrees satisfies <= 45.0 → 10%
    ↳ measurement match: vet=40.0degrees satisfies <= 60.0 → 0%


## 5. Draft the Lay Statement

For the top candidate, gather the veteran's confirmed reports and the DC's criteria, prompt the LLM to draft a one-paragraph Lay Statement in CFR vocabulary, then run the Cypher-backed factuality check. Drafts that introduce content not present in the reports are rejected and regenerated.

In [5]:
from va_agent.output.drafter import draft_lay_statement

top = candidates[0]
draft = draft_lay_statement(driver, user_id, top.code)

print(f'DC {top.code} — Lay Statement draft (attempts={draft.attempts}, factuality={"OK" if draft.factuality.ok else "FAIL"}):\n')
print(draft.text)
print()
if draft.factuality.fabricated_facts:
    print('Fabricated facts caught:', draft.factuality.fabricated_facts)
if draft.factuality.notes:
    print('Notes:', draft.factuality.notes)

DC 5260 — Lay Statement draft (attempts=2, factuality=FAIL):

I have moderate limitation of flexion in my right knee, which is currently measured at 40.0 degrees. During flare-ups, my knee becomes severely limited, and this occurs weekly, lasting for 2-3 days. As a result of my condition, I cannot kneel and I am unable to run more than half a mile.

Fabricated facts caught: [{'value': 2.0, 'unit': '', 'reason': 'no MeasurementReport with this value/unit'}, {'value': 3.0, 'unit': 'days', 'reason': 'no MeasurementReport with this value/unit'}]


## 6. The graph-backed factuality guarantee

Re-run the factuality check directly on the draft and inspect what it found grounded vs. what it would have rejected. Notice the rounded-degree tolerance (±5°) — see `va_agent.retrieval.factuality._is_grounded`.

In [6]:
from va_agent.retrieval.factuality import check_lay_statement

result = check_lay_statement(driver, user_id, draft.text)
print(f'ok: {result.ok}')
print(f'grounded_facts: {result.grounded_facts}')
print(f'fabricated_facts: {result.fabricated_facts}')
print(f'notes: {result.notes}')

ok: False
grounded_facts: [{'value': 40.0, 'unit': 'degrees'}]
fabricated_facts: [{'value': 2.0, 'unit': '', 'reason': 'no MeasurementReport with this value/unit'}, {'value': 3.0, 'unit': 'days', 'reason': 'no MeasurementReport with this value/unit'}]
notes: []


## 7. Cleanup

Reset the persona's user subgraph and close the driver. Re-running the notebook from scratch will work.

In [7]:
reset_user(driver, user_id)
driver.close()
print('Done.')

Done.
